# Road Damage - YOLO Colab Template

This notebook clones the `feature/refactor-training-colab` branch, mounts Google Drive, and runs training, evaluation, and inference using helpers from `src/colab/yolo.py`.

In [5]:
import os
from pathlib import Path

REPO = "road-damage-detection"
# Use the repo URL, not a browser /tree/... link.
REPO_URL = "https://github.com/orzMik/road-damage-detection.git"
BRANCH = "feature/refactor-training-colab"

if Path(REPO).is_dir():
    os.chdir(REPO)
    !git fetch origin
    !git checkout {BRANCH}
else:
    !git clone -b {BRANCH} --single-branch {REPO_URL}
    os.chdir(REPO)

# Install dependencies needed for Colab runs
!pip -q install ultralytics wandb

SyntaxError: invalid syntax (413334625.py, line 8)

In [2]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [3]:
from src.colab.yolo import DriveConfig, ensure_drive_paths, mount_drive, create_yolo_data_yaml

mount_drive()

# Use a shared drive or a personal Drive folder
# Examples:
#   "Shareddrives/<shared_drive_name>/road-damage"
#   "MyDrive/road-damage"
drive_cfg = DriveConfig(drive_root="Shareddrives/<shared_drive_name>/road-damage")
paths = ensure_drive_paths(drive_cfg)

dataset_root = paths.datasets / "processed-yolo"
data_yaml = create_yolo_data_yaml(paths.artifacts / "road_damage_drive.yaml", dataset_root)
data_yaml

ModuleNotFoundError: No module named 'src'

In [ ]:
import os
import wandb
from getpass import getpass

if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass("Enter your W&B API key: ")

# Auth only. Training enables Ultralytics W&B logging via wandb_cfg in train_yolo.
wandb.login(key=os.environ["WANDB_API_KEY"])

In [ ]:
from src.colab.yolo import WandbConfig, train_yolo

# Ultralytics logs train/val losses, mAP, plots, and the best checkpoint to W&B.
wandb_cfg = WandbConfig(
    project="road-damage-classification",
    entity="project-nn",
    name="yolov8n_colab",
    job_type="train",
    config={
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
    },
)

train_out = train_yolo(
    weights="yolov8n.pt",
    data_yaml=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=8,
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab",
    wandb_cfg=wandb_cfg,
)

train_out.save_dir

In [ ]:
from src.colab.yolo import evaluate_yolo

best_weights = train_out.save_dir / "weights" / "best.pt"
metrics = evaluate_yolo(
    weights=best_weights,
    data_yaml=data_yaml,
    split="test",
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab_test",
)
metrics

In [ ]:
from src.colab.yolo import predict_yolo

sample_images = dataset_root / "test" / "images"
predict_results = predict_yolo(
    weights=best_weights,
    source=sample_images,
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab_infer",
    save=True,
)

predict_results[:2]